In [17]:
import pandas as pd
import os

print("Files in raw folder:", os.listdir('../data/raw/'))

movies = pd.read_csv('../data/raw/movies.csv')
ratings = pd.read_csv('../data/raw/ratings.csv')

print("\n--- Movies Data ---")
display(movies.head())

print("\n--- Ratings Data ---")
display(ratings.head())

Files in raw folder: ['movies.csv', 'ratings.csv']

--- Movies Data ---


,movieId,title,genres
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,2,Jumanji (1995),Adventure|Children|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance
4,5,Father of the Bride Part II (1995),Comedy



--- Ratings Data ---


,userId,movieId,rating,timestamp
0,1,16,4.0,1217897793
1,1,24,1.5,1217895807
2,1,32,4.0,1217896246
3,1,47,4.0,1217896556
4,1,50,4.0,1217896523


In [ ]:
print("Movies shape:", movies.shape)      
print("Ratings shape:", ratings.shape)    

print("\nMovies info:")
movies.info()

print("\nRatings info:")
ratings.info()

print("\nMovies describe:")
display(movies.describe(include='all'))

print("\nRatings describe:")
display(ratings.describe())

Movies shape: (10329, 3)
Ratings shape: (105339, 4)

Movies info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10329 entries, 0 to 10328
Data columns (total 3 columns):
 #   Column   Non-Null Count  Dtype 
---  ------   --------------  ----- 
 0   movieId  10329 non-null  int64 
 1   title    10329 non-null  object
 2   genres   10329 non-null  object
dtypes: int64(1), object(2)
memory usage: 242.2+ KB

Ratings info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 105339 entries, 0 to 105338
Data columns (total 4 columns):
 #   Column     Non-Null Count   Dtype  
---  ------     --------------   -----  
 0   userId     105339 non-null  int64  
 1   movieId    105339 non-null  int64  
 2   rating     105339 non-null  float64
 3   timestamp  105339 non-null  int64  
dtypes: float64(1), int64(3)
memory usage: 3.2 MB

Movies describe:


,movieId,title,genres
count,10329.000000,10329,10329
unique,NaN,10327,938
top,NaN,Men with Guns (1997),Drama
freq,NaN,2,1385
mean,31924.282893,NaN,NaN
std,37734.741149,NaN,NaN
min,1.000000,NaN,NaN
25%,3240.000000,NaN,NaN
50%,7088.000000,NaN,NaN
75%,59900.000000,NaN,NaN



Ratings describe:


,userId,movieId,rating,timestamp
count,105339.000000,105339.000000,105339.000000,1.053390e+05
mean,364.924539,13381.312477,3.516850,1.130424e+09
std,197.486905,26170.456869,1.044872,1.802660e+08
min,1.000000,1.000000,0.500000,8.285650e+08
25%,192.000000,1073.000000,3.000000,9.711008e+08
50%,383.000000,2497.000000,3.500000,1.115154e+09
75%,557.000000,5991.000000,4.000000,1.275496e+09
max,668.000000,149532.000000,5.000000,1.452405e+09


In [ ]:
print("\n" + "="*60)
print("DIAGNOSTIC REPORT")
print("="*60)

print(f"Movies — NaN:\n{movies.isnull().sum()}")
print(f"\nRatings — NaN:\n{ratings.isnull().sum()}")

print(f"\nMovies — duplicated movieId: {movies['movieId'].duplicated().sum()}")
print(f"Movies — duplicated title:   {movies['title'].duplicated().sum()}")
print(f"Ratings — duplicated (user,movie): {ratings.duplicated(subset=['userId','movieId']).sum()}")

print(f"\nRatings — rating range: [{ratings['rating'].min()}, {ratings['rating'].max()}]")
print(f"Ratings — rating==0: {(ratings['rating']==0).sum()}")

dup_titles = movies[movies.duplicated(subset='title', keep=False)].sort_values('title')
print(f"\n⚠️  Duplicated titles ({len(dup_titles)}):")
display(dup_titles)

In [ ]:
movies_raw  = movies.copy()
ratings_raw = ratings.copy()

In [ ]:
n_before = len(ratings)
ratings = ratings.dropna(subset=['userId','movieId','rating'])
print(f"Ratings NaN حُذفت: {n_before - len(ratings)}")

movies = movies.dropna(subset=['movieId','title'])
movies['genres'] = movies['genres'].fillna('Unknown')

In [ ]:
ratings['timestamp'] = pd.to_datetime(ratings['timestamp'], unit='s')
ratings['year']  = ratings['timestamp'].dt.year
ratings['month'] = ratings['timestamp'].dt.month

In [ ]:
movies = movies.drop_duplicates(subset='title', keep='first').reset_index(drop=True)

In [ ]:
valid_ids = set(movies['movieId'])
ratings = ratings[ratings['movieId'].isin(valid_ids)].copy()

In [ ]:
ratings = ratings.drop_duplicates(subset=['userId','movieId'], keep='last')

In [ ]:
ratings = ratings[(ratings['rating'] >= 0.5) & (ratings['rating'] <= 5.0)]

In [ ]:
MIN_USER_RATINGS  = 5
MIN_MOVIE_RATINGS = 5

In [ ]:
user_counts = ratings['userId'].value_counts()
before = (ratings['userId'].nunique(), len(ratings))
ratings = ratings[ratings['userId'].isin(user_counts[user_counts >= MIN_USER_RATINGS].index)]
print(f"Users: {before[0]} → {ratings['userId'].nunique()}  "
      f"(حُذف {before[0]-ratings['userId'].nunique()} مستخدم | "
      f"{(1-len(ratings)/before[1])*100:.2f}% من التقييمات)")

In [ ]:
movie_counts = ratings['movieId'].value_counts()
before = (ratings['movieId'].nunique(), len(ratings))
ratings = ratings[ratings['movieId'].isin(movie_counts[movie_counts >= MIN_MOVIE_RATINGS].index)]
print(f"Movies: {before[0]} → {ratings['movieId'].nunique()}  "
      f"(حُذف {before[0]-ratings['movieId'].nunique()} فيلم | "
      f"{(1-len(ratings)/before[1])*100:.2f}% من التقييمات)")

In [ ]:
movies = movies[movies['movieId'].isin(set(ratings['movieId']))].copy()

In [ ]:
genres_dummies = movies['genres'].str.get_dummies(sep='|')
genres_dummies.columns = ['genre_' + c for c in genres_dummies.columns]
print(f"عدد الأجناس الفريدة: {genres_dummies.shape[1]}")

In [ ]:
movie_stats = (ratings.groupby('movieId')['rating']
               .agg(avg_rating='mean', num_ratings='count')
               .reset_index())

In [ ]:
user_stats = (ratings.groupby('userId')['rating']
              .agg(user_avg='mean', user_count='count')
              .reset_index())

In [ ]:
movies_enriched = pd.concat(
    [movies.reset_index(drop=True), genres_dummies.reset_index(drop=True)], axis=1
).merge(movie_stats, on='movieId', how='left')

In [ ]:
# ================================================================
# 🛠️ FEATURE ENGINEERING — إنتاج 3 جداول نظاف
# ================================================================
print("\n" + "="*60)
print("FEATURE ENGINEERING")
print("="*60)

# --- 7.1 أجناس → one-hot ---
genres_dummies = movies['genres'].str.get_dummies(sep='|')
genres_dummies.columns = ['genre_' + c for c in genres_dummies.columns]
print(f"عدد الأجناس الفريدة: {genres_dummies.shape[1]}")

# --- 7.2 إحصاءات الأفلام (نحتاجها باش نضيفها لـ movies_clean) ---
movie_stats = (ratings.groupby('movieId')['rating']
               .agg(avg_rating='mean', num_ratings='count')
               .reset_index())

# --- 7.3 إحصاءات المستخدمين (نحتاجها باش نضيفها لـ ratings_clean) ---
user_stats = (ratings.groupby('userId')['rating']
              .agg(user_avg='mean', user_count='count')
              .reset_index())


# ================================================================
# 📗 الجدول 1: ratings_clean  (تقييمات نظيفة + إحصاءات المستخدم)
# ================================================================
ratings_clean = ratings.merge(user_stats, on='userId', how='left')

print("\n📗 ratings_clean:")
print(f"   shape    : {ratings_clean.shape}")
print(f"   columns  : {ratings_clean.columns.tolist()}")
print(f"   NaN      : {ratings_clean.isnull().sum().sum()}")


# ================================================================
# 📘 الجدول 2: movies_clean  (أفلام نظيفة + أجناس + إحصاءات)
# ================================================================
movies_clean = pd.concat(
    [movies.reset_index(drop=True), genres_dummies.reset_index(drop=True)], axis=1
).merge(movie_stats, on='movieId', how='left')

print("\n📘 movies_clean:")
print(f"   shape    : {movies_clean.shape}")
print(f"   columns  : {movies_clean.columns.tolist()}")
print(f"   NaN      : {movies_clean.isnull().sum().sum()}")


# ================================================================
# 📙 الجدول 3: final_df  (الدمج الكامل)
# ================================================================
final_df = (ratings_clean
            .merge(movies_clean, on='movieId', how='left'))

print("\n📙 final_df (مدموج):")
print(f"   shape    : {final_df.shape}")
print(f"   columns  : {final_df.columns.tolist()}")
print(f"   NaN      : {final_df.isnull().sum().sum()}")


# ================================================================
# 💾 SAVE — نحفظ 3 جداول
# ================================================================
os.makedirs('../data/processed', exist_ok=True)

ratings_clean.to_csv('../data/processed/ratings_clean.csv', index=False)
movies_clean.to_csv('../data/processed/movies_clean.csv',   index=False)
final_df.to_csv('../data/processed/final_df.csv',            index=False)

print("\n✅ Saved:")
print("   ../data/processed/ratings_clean.csv")
print("   ../data/processed/movies_clean.csv")
print("   ../data/processed/final_df.csv")

In [ ]:
summary = pd.DataFrame({
    'Metric': ['Ratings', 'Users', 'Movies', 'Movies rows'],
    'Before': [len(ratings_raw), ratings_raw['userId'].nunique(),
               ratings_raw['movieId'].nunique(), len(movies_raw)],
    'After' : [len(final_df), final_df['userId'].nunique(),
               final_df['movieId'].nunique(), len(movies)],
})
summary['Lost %'] = ((summary['Before'] - summary['After']) / summary['Before'] * 100).round(2)
display(summary)